In [2]:
# ============================================================================
# gen_synth_datasets.py
# Generates synthetic datasets for:
#   1. sc_bus5     (Superconducting 5-qubit multiplexed, T = 500)
#   2. spin_single (Spin-qubit RF reflectometry single, T = 1600)
#   3. spin_bus5   (Spin-qubit 5-channel RF-SET bus, T = 1600)
# ============================================================================
import os
import numpy as np


def _gen_sc_bus5(rng, n_shots=20000, T=500, fs=500e6):
    t = np.arange(T, dtype=np.float64) / fs
    f_if = np.array([25e6, 62e6, 103e6, 141e6, 184e6])
    n_q = 5
    env = 1.0 - np.exp(-t / 80e-9)

    amp = np.array([2600.0, 2500.0, 2700.0, 2450.0, 2550.0])
    theta_0 = np.array([-0.55, -0.50, -0.60, -0.52, -0.58])
    theta_1 = np.array([+0.55, +0.50, +0.60, +0.52, +0.58])

    states = rng.integers(0, 2, size=(n_shots, n_q), dtype=np.uint8)
    sig = np.zeros((n_shots, T), dtype=np.complex128)

    for k in range(n_q):
        phi = np.where(states[:, k:k+1] == 1, theta_1[k], theta_0[k])
        # Non-linear cross-Kerr shift from neighbors k+-1, k+-2
        for offset, coupling in [(-1, 0.14), (1, 0.14), (-2, 0.06), (2, 0.06)]:
            nbr = k + offset
            if 0 <= nbr < n_q:
                phi = phi + coupling * states[:, nbr:nbr+1] * (0.5 + 0.8 * states[:, k:k+1])
        sig += amp[k] * env[None, :] * np.exp(1j * (2.0 * np.pi * f_if[k] * t[None, :] + phi))

    # Weak front-end quadratic + cubic intermodulation
    scale = 12000.0
    sn = sig / scale
    sig = sig + scale * (0.045 * (sn ** 2) + 0.030 * (np.abs(sn) ** 2) * sn)

    # White amplifier noise
    sigma = 1450.0
    sig += rng.normal(0.0, sigma, sig.shape) + 1j * rng.normal(0.0, sigma, sig.shape)
    return sig, states, f_if


def _add_pink_noise(rng, shape, sigma_pink):
    # Simple 1/f approximation via cumulative low-pass filtered white noise
    white = rng.normal(0.0, 1.0, shape) + 1j * rng.normal(0.0, 1.0, shape)
    alpha = 0.04
    pink = np.zeros_like(white)
    acc = np.zeros(shape[0], dtype=np.complex128)
    for n in range(shape[1]):
        acc = (1.0 - alpha) * acc + alpha * white[:, n]
        pink[:, n] = acc
    pink /= (np.std(pink.real) + 1e-12)
    return sigma_pink * pink


def _gen_spin(rng, n_q, n_shots=20000, T=1600, fs=500e6):
    t = np.arange(T, dtype=np.float64) / fs
    f_if_all = np.array([31e6, 71e6, 113e6, 151e6, 191e6])
    f_if = f_if_all[:n_q]
    env = 1.0 - np.exp(-t / 220e-9)  # Slower tank-circuit ring-up (~220 ns)

    amp_0 = np.array([1550.0, 1500.0, 1600.0, 1480.0, 1520.0])[:n_q]
    amp_1 = np.array([2150.0, 2100.0, 2200.0, 2050.0, 2120.0])[:n_q]
    phi_0 = np.array([-0.38, -0.35, -0.40, -0.36, -0.39])[:n_q]
    phi_1 = np.array([+0.42, +0.40, +0.44, +0.39, +0.41])[:n_q]

    states = rng.integers(0, 2, size=(n_shots, n_q), dtype=np.uint8)
    sig = np.zeros((n_shots, T), dtype=np.complex128)

    for k in range(n_q):
        # Model spin T1 relaxation jump during readout (T1 ~ 45 us)
        t_jump = rng.exponential(scale=45e-6, size=(n_shots, 1))
        active_state = np.where((states[:, k:k+1] == 1) & (t[None, :] < t_jump), 1, 0)

        a_t = np.where(active_state == 1, amp_1[k], amp_0[k])
        p_t = np.where(active_state == 1, phi_1[k], phi_0[k])

        # Capacitive inter-dot crosstalk for multi-qubit spin array
        if n_q > 1:
            for offset, c_pull in [(-1, 0.16), (1, 0.16), (-2, 0.05), (2, 0.05)]:
                nbr = k + offset
                if 0 <= nbr < n_q:
                    p_t = p_t + c_pull * states[:, nbr:nbr+1] * (0.4 + 0.9 * active_state)

        sig += a_t * env[None, :] * np.exp(1j * (2.0 * np.pi * f_if[k] * t[None, :] + p_t))

    # Coulomb blockade quadratic curvature + 1/f charge noise + white noise
    scale = 9500.0
    sn = sig / scale
    sig = sig + scale * (0.055 * (sn ** 2))
    sig += _add_pink_noise(rng, sig.shape, sigma_pink=650.0)
    sig += rng.normal(0.0, 1250.0, sig.shape) + 1j * rng.normal(0.0, 1250.0, sig.shape)
    return sig, states, f_if


def save_split(out_dir, name, sig, states, f_if, fs=500e6):
    I_adc = np.clip(np.round(sig.real), -32768, 32767).astype(np.int16)
    Q_adc = np.clip(np.round(sig.imag), -32768, 32767).astype(np.int16)
    X = np.stack([I_adc, Q_adc], axis=-1)
    y = states[:, 0] if states.shape[1] == 1 else states

    n_train = int(0.8 * len(X))
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.npz")
    np.savez_compressed(
        path,
        X_train=X[:n_train], y_train=y[:n_train],
        X_test=X[n_train:],  y_test=y[n_train:],
        f_if=f_if, fs=fs
    )
    print(f"[OK] {path}: X_train={X[:n_train].shape}, y_train={y[:n_train].shape}")


def main():
    rng = np.random.default_rng(2026)
    base = "data/synth"

    sig, st, f_if = _gen_sc_bus5(rng, n_shots=20000, T=500)
    save_split(base, "sc_bus5", sig, st, f_if)

    sig, st, f_if = _gen_spin(rng, n_q=1, n_shots=20000, T=1600)
    save_split(base, "spin_single", sig, st, f_if)

    sig, st, f_if = _gen_spin(rng, n_q=5, n_shots=20000, T=1600)
    save_split(base, "spin_bus5", sig, st, f_if)


if __name__ == "__main__":
    main()

[OK] data/synth\sc_bus5.npz: X_train=(16000, 500, 2), y_train=(16000, 5)
[OK] data/synth\spin_single.npz: X_train=(16000, 1600, 2), y_train=(16000,)
[OK] data/synth\spin_bus5.npz: X_train=(16000, 1600, 2), y_train=(16000, 5)
